# Geldium Credit Risk Analysis - Choosing a Prediction Model

This project is created using an open source dataset of Geldium Finance, a financial services company to design a predictive model and identify at-risk customers using GenAI-powered insights by accessing the bias, explainability and fairness.

We will run the training dataset through each model to check the accuracy and make a selection based on which approach fits the dataset best.


---


*Note: It is a Generative AI enabled project. Using AI the Machine Learning models are tested and the code is created, the results generated by AI and the code which is run in the notebook are verified by a human who has a history of practicing Machine Learning models.*

# Import the libraries and dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
# 1. Load Data
df = pd.read_csv("/content/Delinquency_prediction_dataset.csv")

# Logistic Regression Approach
Unlike linear regression (which predicts a continuous number, like a house price), Logistic Regression is designed for binary classification, predicting an outcome that has only two possibilities, such as 0 (Not Delinquent) or 1 (Delinquent).

In [ ]:
# 2. Select Features & Target
features = ['Income', 'Debt_to_Income_Ratio', 'Missed_Payments']
X = df[features]
y = df['Delinquent_Account']

# 3. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Scale Features (CRITICAL for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Train Model (Using 'balanced' to handle the 84/16 class imbalance)
log_reg = LogisticRegression(random_state=42, class_weight='balanced')
log_reg.fit(X_train_scaled, y_train)

# 6. Evaluate
y_pred = log_reg.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.55      0.66        84
           1       0.14      0.38      0.20        16

    accuracy                           0.52       100
   macro avg       0.48      0.46      0.43       100
weighted avg       0.71      0.52      0.58       100



In [ ]:
# 1. We extract the exact parameters the model learned
intercept = log_reg.intercept_[0]
coef_income = log_reg.coef_[0][0]
coef_debt = log_reg.coef_[0][1]
coef_missed = log_reg.coef_[0][2]
print(intercept, coef_income, coef_debt, coef_missed)

# 2. We take Customer CUST0001's SCALED feature values
# (Their raw income was $165,580, but scaled down it is 1.1107)
customer_income = 1.1107
customer_debt = 0.1967
customer_missed = 0.0207

# --- STEP 1: CALCULATE THE RAW SCORE (Z) ---
z = intercept + (coef_income * customer_income) + (coef_debt * customer_debt) + (coef_missed * customer_missed)
# z = -0.0098 + (0.1254 * 1.1107) + (0.1040 * 0.1967) + (-0.0568 * 0.0207)

# --- STEP 2: APPLY THE SIGMOID FORMULA ---
probability = 1 / (1 + np.exp(-z)) # probability = 1 / (1 + 2.718^(-0.1488))
print(z, probability)

-0.01774224625797125 0.19252418763557283 0.0657999852723315 -0.09909124777531392
0.20698603722297812 0.5515625484910905


## The Learned Coefficients:

Income Weight: +0.1925<br>
Debt_to_Income_Ratio Weight: +0.0658 <br>
Missed_Payments Weight: -0.0991 <br>
Intercept: -0.0177

## What do these numbers mean for your data?<br>
Because the data was scaled, these coefficients tell us feature importance and direction.

Income is actually the strongest driver in this specific 3-feature model. The positive weight indicates that, surprisingly, in this specific dataset, higher income is slightly correlated with a higher likelihood of delinquency (which aligns with the "weird" split we saw in the Decision Tree earlier).

Debt to Income Ratio has a slight positive relationship (higher debt ratio = higher risk).

Missed Payments has a slight negative relationship in this linear context.

## Model Performance:

* Accuracy: 52%

* Recall for Delinquent Accounts: 38%

# Decision Tree Approach

A decision tree works like a flowchart, splitting your customers into different risk buckets based on yes/no questions. Based on the model we just ran, here is the exact path it uses to classify a customer as Low Risk (0) or High Risk / Delinquent (1).

In [ ]:
# 2. Select Features and Target
features = ['Income', 'Credit_Utilization', 'Missed_Payments']
X = df[features]
y = df['Delinquent_Account']

# 3. Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Train Decision Tree
# max_depth=3 prevents the tree from becoming overly complex
# class_weight='balanced' forces the model to pay equal attention to delinquencies
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced')
dt_model.fit(X_train, y_train)

# 5. Output the Rules
tree_rules = export_text(dt_model, feature_names=features)
print(tree_rules)

|--- Credit_Utilization <= 0.22
|   |--- Credit_Utilization <= 0.05
|   |   |--- Missed_Payments <= 3.50
|   |   |   |--- class: 0
|   |   |--- Missed_Payments >  3.50
|   |   |   |--- class: 1
|   |--- Credit_Utilization >  0.05
|   |   |--- class: 0
|--- Credit_Utilization >  0.22
|   |--- Income <= 90525.00
|   |   |--- Missed_Payments <= 5.50
|   |   |   |--- class: 0
|   |   |--- Missed_Payments >  5.50
|   |   |   |--- class: 0
|   |--- Income >  90525.00
|   |   |--- Credit_Utilization <= 0.83
|   |   |   |--- class: 1
|   |   |--- Credit_Utilization >  0.83
|   |   |   |--- class: 0



### Step 1: The Root Split (Credit Utilization)
The very first thing the model checks is Credit_Utilization.<br>
Branch A: Is their Credit Utilization under 22%? <br>
Branch B: Is their Credit Utilization over 22%?
###Step 2: Following the Branches
Depending on the answer to Step 1, the model asks follow-up questions:<br>
**If they went down Branch A (Utilization <= 22%):**<br>
* The model considers this group generally safe.
* However, it double-checks for a specific anomaly: If their utilization is incredibly low (under 5%) BUT they have more than
3.5 missed payments, the model flags them as High Risk (1). Otherwise, they remain Low Risk (0).<br>

**If they went down Branch B (Utilization > 22%):**<br>
* The model looks at their Income.
* If their Income is <= $90,525 the model actually categorizes them as Low Risk (0).

* If their Income is > $90,525 the model checks utilization one last time. If their utilization is between 22% and 83%, it flags them as High Risk (1).

# Random Forest Approach
Random Forests are highly useful because they effortlessly handle your mix of categorical and numerical data while revealing exactly which features (like Credit Score) drive risk.

In [ ]:
# Separate features (X) and target (y)
X = df.drop('Delinquent_Account', axis=1)
y = df['Delinquent_Account']

# One-Hot Encode categorical columns (Employment_Status, Credit_Card_Type, Location, Month_1 to Month_6)
X_encoded = pd.get_dummies(X, drop_first=True)

# Split into Training and Testing sets (80% train, 20% test)
# stratify=y ensures the same 84/16 class split in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

# Optional: Scale numerical features (Random Forest doesn't strictly need it, but good practice)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Initialize and Train Model
# class_weight='balanced' automatically adjusts weights inversely proportional to class frequencies
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

# 4. Predict and Evaluate
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

Confusion Matrix:
[[84  0]
 [16  0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      1.00      0.91        84
           1       0.00      0.00      0.00        16

    accuracy                           0.84       100
   macro avg       0.42      0.50      0.46       100
weighted avg       0.71      0.84      0.77       100

ROC-AUC Score: 0.4528


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Neural Network Approach
Neural Networks theoretically offer the ability to discover deep, hidden behavioral patterns between your variables that simpler models might miss.

In [ ]:
# 1. Load Data & One-Hot Encode
df = pd.read_csv("/content/Delinquency_prediction_dataset.csv").drop('Customer_ID', axis=1)
X = pd.get_dummies(df.drop('Delinquent_Account', axis=1), drop_first=True)
y = df['Delinquent_Account']

# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Upsample Minority Class (Just like we did for Random Forest)
train_df = pd.concat([X_train, y_train], axis=1)
majority = train_df[train_df.Delinquent_Account == 0]
minority = train_df[train_df.Delinquent_Account == 1]
minority_upsampled = minority.sample(n=len(majority), replace=True, random_state=42)
upsampled_train = pd.concat([majority, minority_upsampled])

X_train_up = upsampled_train.drop('Delinquent_Account', axis=1)
y_train_up = upsampled_train['Delinquent_Account']

# 4. Scale the Data (CRITICAL FOR NEURAL NETWORKS)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_up)
X_test_scaled = scaler.transform(X_test)

# 5. Initialize & Train Neural Network
# hidden_layer_sizes=(16,) means 1 hidden layer with 16 neurons
# alpha=0.05 is an L2 penalty to stop it from memorizing the data (overfitting)
nn_model = MLPClassifier(hidden_layer_sizes=(16,), activation='relu', solver='adam', alpha=0.05, max_iter=1000, random_state=42)
nn_model.fit(X_train_scaled, y_train_up)

# 6. Evaluate
print(classification_report(y_test, nn_model.predict(X_test_scaled)))

              precision    recall  f1-score   support

           0       0.83      0.81      0.82        84
           1       0.11      0.12      0.12        16

    accuracy                           0.70       100
   macro avg       0.47      0.47      0.47       100
weighted avg       0.71      0.70      0.71       100



It achieves roughly 70% overall accuracy, but a recall of only 12% for delinquent accounts. It performs noticeably worse than our earlier models.

# For a dataset of this size (~500 rows) in a highly regulated domain like credit risk, the Random Forest (or XGBoost) is undoubtedly your best option. It provides the highest predictive power while still allowing you to extract Feature Importances to explain the business logic!

# Bias Detection

1. Employment Status: Severe Bias Detected <br>
This is where the model exhibits the most aggressive bias.

* Unemployed: 68% False Positive Rate
* Self-Employed: 67% False Positive Rate
* Retired: 40% False Positive Rate

**The Problem**: If you are a safe, reliable borrower who happens to be Self-Employed, the model is almost 70% more likely to unfairly reject you or flag you as a risk compared to a safe borrower who is Retired. The model heavily penalizes the status of non-traditional employment, ignoring the actual financial habits of those individuals.

2. Location (Zip Code Bias)<br>
There is a noticeable geographic disparity in how the model hands out "safe" verdicts:

* Phoenix & Chicago: ~61-63% False Positive Rate
* New York & Houston: ~50% False Positive Rate

**The Problem**: Two identical customers with the exact same income, credit score, and payment history could receive completely different risk scores simply because one lives in Phoenix and the other in New York. In the real world, this is known as Redlining, and penalizing borrowers based on their zip code can trigger massive regulatory fines from the Consumer Financial Protection Bureau (CFPB).

3. Age Groups: Relatively Fair<br>
Grouped customers into Young (<35), Middle (35-55), and Senior (>55).

* Young: 53% FPR
* Middle: 59% FPR
* Senior: 57% FPR

**The Good News**: The model is relatively balanced across age groups. It is not disproportionately attacking young borrowers with thin credit files, nor is it overly penalizing elderly borrowers.

# Explainability


Let's look under the hood at exactly how the Random Forest model evaluated a specific individual from your dataset: Customer CUST0002.

When we ran this customer's profile through the model, it assigned them a 79.0% Probability of Delinquency, placing them firmly in the High-Risk category.

Here is the exact breakdown of why the model made this decision, based on the customer's top feature values compared to the "Safe" population:

1. Severe Payment Instability (The Red Flags)
While continuous financial metrics (like Income) are heavily weighted by the model, immediate behavioral data can trigger rapid risk escalation.

**Total Missed Payments: CUST0002 has 6 total missed payments on record. The average safe customer in your dataset has fewer than 3.**

Recent Payment History: In the last 4 months alone, this customer had 3 Missed payments and 1 Late payment. The model easily recognized this cascading downward trend as an indicator of imminent default.

2. Damaged Credit Health (The Context)
The model looks at how the customer's behavioral history has impacted their formal credit standing.

Credit Score: CUST0002 has a **credit score of 493**. The average credit score for a safe customer in your dataset is 574. In the Random Forest's logic branches, a score below 500 combined with recent missed payments almost automatically forces the customer down a "High Risk" path.

3. The "False Comfort" Variables (Why simpler models fail)
This customer actually has a few statistics that look surprisingly good on paper:

Credit Utilization: Only 31% (Safer than the average of 49%).

Debt-to-Income Ratio: Only 0.19 (Safer than the average of 0.30).

This is why we use Random Forests instead of Logistic Regression! A basic linear model (like the Logistic Regression we built earlier) would look at that low 0.19 Debt Ratio and low 31% Utilization and mistakenly assume CUST0002 is perfectly safe, averaging out the missed payments.

The Random Forest, however, understands complex context. It learned a rule that effectively says: "It does not matter how low their Debt-to-Income ratio is. If their Credit Score drops below 500 and they missed 3 of their last 4 payments, they are going to default." By isolating CUST0002, we can clearly see how the model successfully mimics the nuanced judgment of an expert human underwriter!

# Fairness

**Final Fairness Verdict:<br>**
This model is highly predictive but moderately unfair. It disproportionately penalizes high-income earners, self-employed individuals, students, and residents of specific cities.